# A1 — Identificación causal: programa de empleo (Jobs / LaLonde-style)

**Curso:** IA Causal en Producción (ECO-490 / ECON-540) · *enfoque profesional*  
**Sesiones:** S4–S6 · **Módulo 1**  
**Lectura monografía (PDF 32 caps.):** Cap. **4** Ingeniería de datos · Cap. **6** Regularización · Cap. **7** Datos de entrenamiento · (Cap. **16** PDS extendido ◇ opcional)  
**Labs:** L04 · L05 · L06  
**Dataset:** `real/jobs` (`scenario_id`: `real_jobs`)  
**Técnica:** pregunta causal, ATE, DAG, overlap, diff-in-means, MCO, Post-Double-Selection  
**Alimenta:** base conceptual del hilo CDS (no artefactos B)

### Pregunta de valor (disrupción)

> *Si solo predijéramos ingresos post-programa, ¿a quién “recomendaríamos” el entrenamiento?*  
> *Si estimamos el **efecto** del programa, ¿sigue siendo un go/no-go o cambia la decisión?*

### Objetivos de aprendizaje

1. Distinguir pregunta **predictiva** vs. **causal** en un programa laboral.
2. Declarar el estimando **ATE** y los supuestos de identificación (ignorabilidad, overlap, SUTVA).
3. Diagnosticar soporte común (propensity, balance).
4. Comparar: diferencia de medias, OLS y Post-Double-Selection (PDS).
5. Interpretar **sin** ground truth en el cuerpo principal (truth solo al final / autograder).


### Decisión de negocio

Esta es la pregunta de valor de arriba, en los términos en los que la vería un patrocinador del programa: *si solo predijéramos ingresos, priorizaríamos por perfil; si estimamos el efecto, la decisión de negocio es un go/no-go de inversión* — y ese es el estimando (ATE) que se declara y defiende en este notebook.

## Aviso sobre los datos (transparencia)

Este notebook usa un escenario del **Causal Benchmark Ecosystem**.

Los archivos en `datasets/real/` **no son datos clínicos ni administrativos reales**.  
Lo auténtico de la literatura son **nombres de variables, diseño del estudio y órdenes de magnitud de referencia** (p. ej. ATE de programas tipo NSW / LaLonde).  
Los valores numéricos se generan con SEM / fórmulas documentadas para que exista **ground truth** (`truth_*.csv`) usable en evaluación y en el autograder.

| Capa | Archivos | Quién la ve en este notebook |
|------|----------|------------------------------|
| Observables | `observed_*.csv` | **Siempre** (flujo del estudiante) |
| Ground truth | `truth_*.csv` (`y0`, `y1`, `tau`) | **Solo** sección final marcada «evaluación con truth» |
| Identificación / scorecard | `identification.json`, `scorecard.json` | Lectura guiada; no copiar ciegamente |

Consulta `metadata.json` → campos `what_is_real` / `what_is_synthetic`.  
**Política:** no uses `y0`, `y1` ni `tau` fuera de la sección final de evaluación.


## 1. Setup y carga del escenario

Semilla global del curso: `RANDOM_STATE = 42`.  
Los splits son los **oficiales** del escenario (`observed_train/validation/test`); no re-particionamos.


In [ ]:
# Paths relativos al monorepo (kernel-less: funciona con nbconvert/pytest)
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Package root = parent of notebooks/
_NOTEBOOK_DIR = Path.cwd()
# When executed from notebooks/, parent is package; also try common alternatives
_CANDIDATES = [
    _NOTEBOOK_DIR.parent,  # .../03_notebooks_caso_estudio if cwd is notebooks/
    _NOTEBOOK_DIR,         # if cwd is already package root
    Path(__file__).resolve().parent.parent if "__file__" in dir() else None,
]
for _p in _CANDIDATES:
    if _p is None:
        continue
    if (_p / "lib" / "paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        PACKAGE_ROOT = _p
        break
else:
    # Fallback: walk up from cwd
    here = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for parent in [here, *here.parents]:
        if (parent / "lib" / "paths.py").is_file():
            sys.path.insert(0, str(parent))
            PACKAGE_ROOT = parent
            break
    if PACKAGE_ROOT is None:
        raise RuntimeError(
            "No se encontró lib/paths.py. Ejecuta desde "
            "03_notebooks_caso_estudio/ o notebooks/."
        )

from lib.paths import (  # noqa: E402
    RANDOM_STATE,
    SCENARIO_A1,
    SCENARIO_IDS,
    data_root,
    package_root,
)
from lib.data_io import (  # noqa: E402
    assert_no_truth_columns,
    get_ytx,
    load_scenario,
)
from lib.identification import (  # noqa: E402
    balance_table,
    diff_in_means,
    fit_propensity,
    ols_ate,
    overlap_diagnostics,
    post_double_selection_ate,
    trim_by_propensity,
)
from lib.metrics import ate_absolute_error, summarize_ate_comparison  # noqa: E402
from lib.plotting import (  # noqa: E402
    apply_course_style,
    plot_propensity_overlap,
    plot_smd_balance,
)

np.random.seed(RANDOM_STATE)
apply_course_style()

print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"PACKAGE_ROOT = {package_root()}")
print(f"DATA_ROOT    = {data_root()}")
print(f"SCENARIO_A1  = {SCENARIO_A1}  → {SCENARIO_IDS[SCENARIO_A1]}")


In [ ]:
# Carga SOLO observables + sidecars (sin truth)
bundle = load_scenario(SCENARIO_A1)
assert_no_truth_columns(bundle.train, context="train")
assert_no_truth_columns(bundle.test, context="test")

print("scenario_id :", bundle.scenario_id)
print("shapes      : train", bundle.train.shape,
      "| val", bundle.validation.shape,
      "| test", bundle.test.shape)
print("columnas    :", list(bundle.train.columns))

# Transparencia desde metadata (no usar ate_true para «chequear» hasta el final)
meta = bundle.meta
print("\n--- Transparencia (metadata) ---")
print("name           :", meta.get("name"))
print("description    :", meta.get("description"))
print("data_origin    :", meta.get("data_origin"))
print("what_is_real   :", meta.get("what_is_real"))
print("what_is_synthetic:", meta.get("what_is_synthetic"))
print("note           :", meta.get("note"))
print("reference      :", meta.get("reference"))
# ate_true existe en metadata; NO lo usamos en estimación del estudiante
print("(ate_true en metadata está reservado a evaluación final / autograder)")


In [ ]:
# Roles de columnas y matrices de trabajo (train = estimación principal)
Y_COL = bundle.outcome_column()   # re78
T_COL = bundle.treatment_column()  # treatment
X_COLS = bundle.covariate_columns()

print(f"Y = {Y_COL!r}  |  T = {T_COL!r}")
print(f"X ({len(X_COLS)}): {X_COLS}")

Y_tr, T_tr, X_tr = bundle.ytx("train")
Y_te, T_te, X_te = bundle.ytx("test")

# Muestra de trabajo: train (estimación) — test se usa para comparación / evaluación
df_train = bundle.train.copy()
df_test = bundle.test.copy()


## 2. EDA causal: tratados vs. controles y balance

Antes de estimar, describimos la **asignación** del tratamiento y el **balance** de covariables pre-tratamiento mediante la diferencia estandarizada de medias (SMD).  
Regla práctica: \(|\mathrm{SMD}| > 0.1\) sugiere desbalance relevante.


In [ ]:
def describe_by_treatment(df: pd.DataFrame, y_col: str, t_col: str, x_cols: list) -> pd.DataFrame:
    """Tabla descriptiva: n, tasa de tratamiento, medias de Y y covariables por brazo."""
    rows = []
    for t_val, label in [(1, "Tratado"), (0, "Control")]:
        sub = df[df[t_col] == t_val]
        row = {
            "grupo": label,
            "n": len(sub),
            "fracción": len(sub) / len(df),
            f"mean_{y_col}": sub[y_col].mean(),
            f"std_{y_col}": sub[y_col].std(),
        }
        for c in x_cols:
            row[f"mean_{c}"] = sub[c].mean()
        rows.append(row)
    return pd.DataFrame(rows)


desc = describe_by_treatment(df_train, Y_COL, T_COL, X_COLS)
print("=== Descriptivos train (por brazo) ===")
display_cols = ["grupo", "n", "fracción", f"mean_{Y_COL}", f"std_{Y_COL}"]
print(desc[display_cols].to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print("\nMedias de covariables:")
print(desc[["grupo"] + [f"mean_{c}" for c in X_COLS]].to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print(f"\nTasa de tratamiento (train): {T_tr.mean():.3f}")


In [ ]:
# Balance: SMD por covariable
bal = balance_table(X_tr, T_tr, covariates=X_COLS)
print("=== Balance de covariables (SMD) — train ===")
print(bal.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nMáx |SMD|: {bal['abs_smd'].max():.4f}")
print(f"Covariables con |SMD| > 0.1: {(bal['abs_smd'] > 0.1).sum()} / {len(bal)}")

fig, ax = plt.subplots()
plot_smd_balance(bal, ax=ax, title="Balance de covariables (SMD) — Jobs train")
plt.tight_layout()
plt.show()


## 3. Pregunta causal y estimando

### Plantilla (rellena / justifica)

| Elemento | Declaración |
|----------|-------------|
| **Unidades** | Individuos en edad laboral del escenario Jobs (LaLonde-style sintético; n≈1722) |
| **Tratamiento** \(T\) | Participación en programa de formación laboral (`treatment` ∈ {0,1}) |
| **Outcome** \(Y\) | Ingresos post-programa en 1978 (`re78`, dólares) |
| **Población** | Población del escenario (mezcla semántica NSW experimental + controles observacionales sintéticos) |
| **Estimando** | **ATE** \(= \mathbb{E}[Y(1) - Y(0)]\): efecto medio del programa sobre ingresos si *todos* fueran tratados vs. *nadie* |

### Predictivo vs. causal

- **Predictivo:** ¿Qué ingresos esperamos para alguien con perfil \(x\) que ya entró al programa? \(\rightarrow \mathbb{E}[Y \mid T=1, X=x]\).
- **Causal:** ¿Cuánto *cambian* los ingresos *por causa* del programa? \(\rightarrow \mathbb{E}[Y(1)-Y(0)]\).

La segunda requiere supuestos de identificación; la primera no identifica un efecto de intervención.


## 4. DAG supuesto y supuestos de identificación

### Diagrama (Mermaid)

```mermaid
flowchart LR
  age[age] --> T[treatment]
  educ[educ] --> T
  black[black] --> T
  hispan[hispan] --> T
  married[married] --> T
  nodegree[nodegree] --> T
  re74[re74] --> T
  re75[re75] --> T
  age --> Y[re78]
  educ --> Y
  black --> Y
  hispan --> Y
  married --> Y
  nodegree --> Y
  re74 --> Y
  re75 --> Y
  T --> Y
```

**Lectura:** las 8 covariables pre-tratamiento confunden \(T\) y \(Y\); condicional en \(X\), no hay confusores latentes *en el diseño del SEM del benchmark*.


In [ ]:
# DAG con networkx (exportable / auditable)
import networkx as nx

def build_jobs_dag(covariates):
    """DAG de identificación para Jobs: X → T, X → Y, T → Y."""
    G = nx.DiGraph()
    G.add_node("treatment")
    G.add_node("re78")
    for c in covariates:
        G.add_node(c)
        G.add_edge(c, "treatment")
        G.add_edge(c, "re78")
    G.add_edge("treatment", "re78")
    return G


G = build_jobs_dag(X_COLS)
print(f"Nodos ({G.number_of_nodes()}): {sorted(G.nodes())}")
print(f"Aristas ({G.number_of_edges()}):")
for u, v in sorted(G.edges()):
    print(f"  {u} → {v}")

# Layout simple para visualización
fig, ax = plt.subplots(figsize=(9, 5))
pos = {}
# Covariates on left, T center, Y right
for i, c in enumerate(X_COLS):
    pos[c] = (0.0, float(len(X_COLS) - 1 - i))
pos["treatment"] = (1.5, (len(X_COLS) - 1) / 2)
pos["re78"] = (3.0, (len(X_COLS) - 1) / 2)
nx.draw_networkx(
    G,
    pos=pos,
    ax=ax,
    node_color=["#DD8452" if n == "treatment" else "#55A868" if n == "re78" else "#4C72B0" for n in G.nodes()],
    node_size=1200,
    font_size=8,
    arrows=True,
    arrowsize=12,
    edge_color="#555555",
)
ax.set_title("DAG supuesto — Jobs (ajuste por X pre-tratamiento)")
ax.axis("off")
plt.tight_layout()
plt.show()


### Supuestos de identificación (ATE bajo selección en observables)

1. **SUTVA / no interferencia:** el tratamiento de \(i\) no afecta el outcome de \(j\); una sola versión del programa.
2. **Ignorabilidad condicional (unconfoundedness):**  
   \(\{Y(0), Y(1)\} \perp T \mid X\)  
   con \(X =\) adjustment set (las 8 covariables pre-tratamiento).
3. **Overlap (positividad):** \(0 < e(x) = P(T=1\mid X=x) < 1\) para casi todo \(x\) en la población de interés.
4. **Medición correcta:** \(T\), \(Y\) y \(X\) medidos sin error diferencial relevante (en el SEM: por construcción).

> **Importante:** `identification.json` del benchmark es una **anotación de diseño del SEM**, no una prueba empírica de ignorabilidad en el mundo real.


In [ ]:
# Lectura guiada de identification.json (no copiar ciegamente)
ident = bundle.identification
print("scenario_id            :", ident.get("scenario_id"))
print("identification_status  :", ident.get("identification_status"))
print("valid_adjustment_sets  :", ident.get("valid_adjustment_sets"))
print("assumptions:")
for a in ident.get("assumptions", []):
    print("  -", a)
print("notes:")
for n in ident.get("notes", []):
    print("  -", n)

adjustment_set = list(ident.get("valid_adjustment_sets", [[]])[0])
print("\nAdjustment set (benchmark):", adjustment_set)
print("Coincide con X del loader :", set(adjustment_set) == set(X_COLS))


### Reflexión: por diseño del SEM vs. paper con datos reales

| | **Por construcción del simulador** | **En un paper con NSW/CPS reales** |
|--|--------------------------------------|-------------------------------------|
| Ignorabilidad | Todas las confusoras del SEM están en \(X\) | Hay que argumentar (y dudar) de confusión residual (motivación, redes, etc.) |
| Adjustment set | Las 8 covars del metadata | Mismo set *clásico*, pero no garantizado suficiente |
| Validez externa | Limitada: SEM no es la población de 1970s EE.UU. | También limitada (selección NSW, CPS) |

**Tarea:** en la sección de escritura (§9) justifica por qué el adjustment set es defendible *aquí* y qué arriesgarías en el mundo real.


## 5. Diagnósticos de overlap (propensity score)

Ajustamos un modelo logístico \(e(x) = P(T=1\mid X=x)\), inspeccionamos el soporte común y aplicamos recorte opcional en \([0.02, 0.98]\).


In [ ]:
# Propensity en train
prop_model, e_tr = fit_propensity(X_tr, T_tr, random_state=RANDOM_STATE)
ov = overlap_diagnostics(e_tr, T_tr, trim_low=0.02, trim_high=0.98)

print("=== Overlap diagnostics (train) ===")
for k, v in ov.items():
    if isinstance(v, float):
        print(f"  {k:28s}: {v:.6f}")
    else:
        print(f"  {k:28s}: {v}")

frac_interior = ov["frac_strictly_interior"]
print(f"\nFracción con 0 < e(x) < 1: {frac_interior:.4f}")
assert frac_interior >= 0.95, "Overlap insuficiente: <95% de unidades con propensity interior"

fig, ax = plt.subplots()
plot_propensity_overlap(e_tr, T_tr, ax=ax, title="Overlap del propensity score — Jobs train")
ax.axvline(0.02, color="gray", ls="--", lw=1, label="trim 0.02/0.98")
ax.axvline(0.98, color="gray", ls="--", lw=1)
ax.legend()
plt.tight_layout()
plt.show()

# Recorte (diagnóstico; estimaciones principales usan muestra completa en este notebook)
df_trim, mask_trim = trim_by_propensity(df_train, e_tr, trim_low=0.02, trim_high=0.98)
print(f"Unidades tras trim [0.02, 0.98]: {mask_trim.sum()} / {len(mask_trim)} "
      f"(descartadas: {(~mask_trim).sum()}, {100*(~mask_trim).mean():.2f}%)")


## 6. Estimación: diferencia de medias y OLS

### Diferencia de medias (naive)

\[
\hat{\tau}_{\mathrm{DIM}} = \bar{Y}_{T=1} - \bar{Y}_{T=0}
\]

Es el ATE solo si el tratamiento es aleatorizado (o no hay confusión). Con desbalance de \(X\), es un estimador **sesgado** del ATE.

### OLS ajustado

\[
Y = \alpha + \tau T + X^\top\beta + \varepsilon
\]

El coeficiente \(\hat\tau\) es el ATE bajo linealidad + ignorabilidad condicional (o una proyección lineal del efecto).


In [ ]:
# --- Estimadores en TRAIN (flujo estudiante; sin truth) ---
ate_dim_train = diff_in_means(Y_tr, T_tr)
ate_ols_train = ols_ate(Y_tr, T_tr, X_tr)

print("=== Estimación en train (sin truth) ===")
print(f"Diff-in-means : {ate_dim_train:,.4f}")
print(f"OLS (T + 8 X) : {ate_ols_train:,.4f}")

# También reportamos en test (misma especificación, sin usar truth)
ate_dim_test = diff_in_means(Y_te, T_te)
ate_ols_test = ols_ate(Y_te, T_te, X_te)
print("\n=== Misma especificación en test (sin truth) ===")
print(f"Diff-in-means : {ate_dim_test:,.4f}")
print(f"OLS (T + 8 X) : {ate_ols_test:,.4f}")


# --- Error estándar e IC 95% (sin tocar los puntos ATE de arriba) ---
# diff-in-means: SE de Welch; OLS: SE HC1 sobre el coeficiente del tratamiento.
from scipy import stats as _sst


def _dim_se_ci(y, t, alpha=0.05):
    ya = np.asarray(y, dtype=float)
    ta = np.asarray(t, dtype=int)
    yt, yc = ya[ta == 1], ya[ta == 0]
    d = float(yt.mean() - yc.mean())
    se = float(np.sqrt(yt.var(ddof=1) / yt.size + yc.var(ddof=1) / yc.size))
    z = float(_sst.norm.ppf(1 - alpha / 2))
    return d, se, d - z * se, d + z * se


def _ols_se_ci(y, t, X, alpha=0.05):
    import statsmodels.api as _sm

    design = np.column_stack([np.asarray(t, dtype=float), np.asarray(X, dtype=float)])
    res = _sm.OLS(np.asarray(y, dtype=float), _sm.add_constant(design)).fit(
        cov_type="HC1"
    )
    coef, se = float(res.params[1]), float(res.bse[1])
    lo, hi = res.conf_int(alpha=alpha)[1]
    return coef, se, float(lo), float(hi)


_, se_dim_train, ci_lo_dim_train, ci_hi_dim_train = _dim_se_ci(Y_tr, T_tr)
_, se_ols_train, ci_lo_ols_train, ci_hi_ols_train = _ols_se_ci(Y_tr, T_tr, X_tr)
_, se_dim_test, ci_lo_dim_test, ci_hi_dim_test = _dim_se_ci(Y_te, T_te)
_, se_ols_test, ci_lo_ols_test, ci_hi_ols_test = _ols_se_ci(Y_te, T_te, X_te)

print("\n=== SE / IC 95% (train) ===")
print(
    f"Diff-in-means : {ate_dim_train:,.4f}  SE={se_dim_train:,.4f}  "
    f"IC=[{ci_lo_dim_train:,.4f}, {ci_hi_dim_train:,.4f}]"
)
print(
    f"OLS (T + 8 X) : {ate_ols_train:,.4f}  SE={se_ols_train:,.4f}  "
    f"IC=[{ci_lo_ols_train:,.4f}, {ci_hi_ols_train:,.4f}]"
)


## 7. Post-Double-Selection (PDS)

Idea (Belloni, Chernozhukov, Hansen): cuando hay muchos controles (o una expansión flexible de \(X\)),

1. **Lasso** de \(Y \sim X\) → controles \(S_Y\)
2. **Lasso** de \(T \sim X\) → controles \(S_T\)
3. **OLS** de \(Y \sim T + X_{S_Y \cup S_T}\)

Así se seleccionan confusores que predicen el outcome *o* el tratamiento (doble selección), reduciendo sesgo de omisión selectiva.

Expandimos features ligeras (polinomios / interacciones de `age`, `educ`, `re74`, `re75`) para que Lasso tenga algo que seleccionar.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures


def expand_jobs_features(
    X: pd.DataFrame,
    continuous=None,
    degree: int = 2,
    interaction_only: bool = False,
) -> pd.DataFrame:
    """Expansión polinómica ligera de covariables continuas + resto de controles.

    Parameters
    ----------
    X:
        DataFrame de covariables base.
    continuous:
        Columnas a expandir con PolynomialFeatures. Default: age, educ, re74, re75
        si existen.
    degree:
        Grado polinómico (2 = cuadrados + interacciones).
    interaction_only:
        Si True, solo interacciones (sin potencias puras).
    """
    cont = continuous or [c for c in ("age", "educ", "re74", "re75") if c in X.columns]
    other = [c for c in X.columns if c not in cont]
    parts = []
    if cont:
        poly = PolynomialFeatures(
            degree=degree,
            include_bias=False,
            interaction_only=interaction_only,
        )
        Xp = poly.fit_transform(X[cont].to_numpy(dtype=float))
        names = poly.get_feature_names_out(cont)
        parts.append(pd.DataFrame(Xp, columns=names, index=X.index))
    if other:
        parts.append(X[other].copy())
    if not parts:
        return X.copy()
    out = pd.concat(parts, axis=1)
    # Evitar nombres duplicados
    out = out.loc[:, ~out.columns.duplicated()]
    return out


X_tr_exp = expand_jobs_features(X_tr)
X_te_exp = expand_jobs_features(X_te)
print(f"Features base : {X_tr.shape[1]} → expandidas: {X_tr_exp.shape[1]}")
print("Nombres (expand):", list(X_tr_exp.columns)[:12], "...")

pds_train = post_double_selection_ate(
    Y_tr, T_tr, X_tr_exp, random_state=RANDOM_STATE
)
ate_pds_train = float(pds_train["ate"])
print("\n=== PDS (train, features expandidas) ===")
print(f"ATE PDS     : {ate_pds_train:,.4f}")
print(f"n_selected  : {pds_train['n_selected']}")
print(f"selected    : {pds_train['selected']}")
print(f"selected_y  : {pds_train['selected_y']}")
print(f"selected_t  : {pds_train['selected_t']}")

pds_test = post_double_selection_ate(
    Y_te, T_te, X_te_exp, random_state=RANDOM_STATE
)
ate_pds_test = float(pds_test["ate"])
print("\n=== PDS (test, features expandidas) ===")
print(f"ATE PDS     : {ate_pds_test:,.4f}")
print(f"selected    : {pds_test['selected']}")

# PDS también con las 8 covars sin expandir (referencia legible)
pds_base = post_double_selection_ate(Y_tr, T_tr, X_tr, random_state=RANDOM_STATE)
print(f"\nPDS sin expansión (train): ATE={pds_base['ate']:,.4f}, selected={pds_base['selected']}")


# --- SE / IC 95% para PDS (OLS final sobre T + controles seleccionados, HC1) ---
def _pds_se_ci(y, t, X_exp, selected, alpha=0.05):
    import statsmodels.api as _sm

    if selected:
        Xs = np.asarray(X_exp[list(selected)], dtype=float)
    else:
        Xs = np.empty((len(np.asarray(y)), 0))
    design = np.column_stack([np.asarray(t, dtype=float), Xs])
    res = _sm.OLS(np.asarray(y, dtype=float), _sm.add_constant(design)).fit(
        cov_type="HC1"
    )
    coef, se = float(res.params[1]), float(res.bse[1])
    lo, hi = res.conf_int(alpha=alpha)[1]
    return coef, se, float(lo), float(hi)


_, se_pds_train, ci_lo_pds_train, ci_hi_pds_train = _pds_se_ci(
    Y_tr, T_tr, X_tr_exp, pds_train["selected"]
)
_, se_pds_test, ci_lo_pds_test, ci_hi_pds_test = _pds_se_ci(
    Y_te, T_te, X_te_exp, pds_test["selected"]
)
_, se_pds_base, ci_lo_pds_base, ci_hi_pds_base = _pds_se_ci(
    Y_tr, T_tr, X_tr, pds_base["selected"]
)

print("\n=== SE / IC 95% PDS (train) ===")
print(
    f"PDS expandido : {ate_pds_train:,.4f}  SE={se_pds_train:,.4f}  "
    f"IC=[{ci_lo_pds_train:,.4f}, {ci_hi_pds_train:,.4f}]"
)
print(
    f"PDS base (8 X): {float(pds_base['ate']):,.4f}  SE={se_pds_base:,.4f}  "
    f"IC=[{ci_lo_pds_base:,.4f}, {ci_hi_pds_base:,.4f}]"
)


## 8. Tabla comparativa y scorecard del curso

El `scorecard.json` del escenario es un **piso de referencia** del monorepo (baselines del release), no la «respuesta correcta» a copiar. Comparamos nuestros estimadores en train/test con esos baselines.


In [ ]:
# Baselines del scorecard (piso del curso)
sc = bundle.scorecard.get("baseline_results", {})
scorecard_rows = []
for key, val in sc.items():
    if not isinstance(val, dict):
        continue
    if "ate_estimate" in val:
        scorecard_rows.append({
            "fuente": "scorecard",
            "estimator": key,
            "ate_hat": val["ate_estimate"],
            "abs_error_vs_truth_sc": val.get("absolute_error"),
        })

scorecard_df = pd.DataFrame(scorecard_rows)
print("=== Scorecard baselines (reference floor) ===")
print(scorecard_df.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

# Nuestros estimadores (sin truth)
estimates_train = {
    "diff_in_means": ate_dim_train,
    "ols_adjusted": ate_ols_train,
    "pds_expanded": ate_pds_train,
    "pds_base_x": float(pds_base["ate"]),
}
estimates_test = {
    "diff_in_means": ate_dim_test,
    "ols_adjusted": ate_ols_test,
    "pds_expanded": ate_pds_test,
}

# SE e IC 95% por estimador/split (calculados en §6 y §7; HC1 en OLS/PDS)
se_ci_map = {
    ("train", "diff_in_means"): (se_dim_train, ci_lo_dim_train, ci_hi_dim_train),
    ("train", "ols_adjusted"): (se_ols_train, ci_lo_ols_train, ci_hi_ols_train),
    ("train", "pds_expanded"): (se_pds_train, ci_lo_pds_train, ci_hi_pds_train),
    ("train", "pds_base_x"): (se_pds_base, ci_lo_pds_base, ci_hi_pds_base),
    ("test", "diff_in_means"): (se_dim_test, ci_lo_dim_test, ci_hi_dim_test),
    ("test", "ols_adjusted"): (se_ols_test, ci_lo_ols_test, ci_hi_ols_test),
    ("test", "pds_expanded"): (se_pds_test, ci_lo_pds_test, ci_hi_pds_test),
}

cmp_train = summarize_ate_comparison(estimates_train)
cmp_train.insert(0, "split", "train")
cmp_test = summarize_ate_comparison(estimates_test)
cmp_test.insert(0, "split", "test")
comparison = pd.concat([cmp_train, cmp_test], ignore_index=True)

# Sumamos SE e IC a la tabla comparativa (sin tocar ate_hat)
comparison["se"] = [se_ci_map[(r.split, r.estimator)][0] for r in comparison.itertuples()]
comparison["ci_low"] = [se_ci_map[(r.split, r.estimator)][1] for r in comparison.itertuples()]
comparison["ci_high"] = [se_ci_map[(r.split, r.estimator)][2] for r in comparison.itertuples()]

print("\n=== Nuestros estimadores (sin truth) ===")
print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:,.4f}",
    )
)

# Alinear OLS train con scorecard OLS (debe ser del mismo orden)
ols_sc = sc.get("inference_ols_adjusted", {}).get("ate_estimate")
if ols_sc is not None:
    print(f"\n|OLS_train - OLS_scorecard| = {abs(ate_ols_train - ols_sc):.6f}")


## 9. Justificación escrita de la identificación (obligatoria)

**Extensión orientativa:** ~½ página. No dejes esta celda vacía: el autograder comprueba longitud mínima del texto de identificación.

### Respuesta de referencia (modelo de escritura)

**Pregunta e estimando.** Estimamos el ATE del programa de formación laboral sobre ingresos de 1978 en la población del escenario Jobs. El estimando es \(\mathbb{E}[Y(1)-Y(0)]\), no un efecto condicional en tratados (ATT) ni una predicción de \(Y\mid T,X\).

**Identificación.** Bajo SUTVA, ignorabilidad condicional en el adjustment set de ocho covariables pre-tratamiento (`age`, `educ`, `black`, `hispan`, `married`, `nodegree`, `re74`, `re75`) y overlap, el ATE es identificable por ajuste (regresión / reponderación / doubly robust). El DAG supone que no hay confusores latentes abiertos tras condicionar en \(X\).

**Por diseño del SEM vs. mundo real.** En el benchmark, `identification_status` declara identificabilidad *por construcción* (todas las confusoras simuladas son observables). En un estudio real estilo LaLonde/Dehejia–Wahba, el mismo set es el *candidato clásico*, pero la ignorabilidad es una hipótesis empírica frágil: motivación no observada, historial laboral detallado y redes locales pueden sesgar el ATE.

**Diagnósticos.** Observamos desbalance moderado (\(|\mathrm{SMD}|\) hasta ~0.19 en `nodegree`/`married`/`age`) y buen overlap: propensity en un rango interior (~0.09–0.53 en train), con ~0% de unidades fuera de \([0.02, 0.98]\). El recorte apenas descarta masa; el soporte común es usable.

**Estimación.** La diferencia de medias y el OLS ajustado dan ATE del orden de ~1 600 USD en train, coherente con el scorecard del curso. El PDS con expansión polinómica selecciona un subconjunto de controles (y términos no lineales) y produce un punto cercano al OLS, lo que sugiere que la especificación lineal ya captura la mayor parte del ajuste necesario en este SEM.

**Riesgo de validez externa.** Aunque el ATE interno del SEM sea recuperable, **no** se generaliza automáticamente a programas laborales contemporáneos: el SEM fija una magnitud de referencia (~1 600–1 800 USD) y una estructura lineal de confusión, no la heterogeneidad institucional real del NSW ni del mercado laboral actual.

### Plantilla para tu entrega (si reescribes)

1. Unidades, \(T\), \(Y\), población, estimando (ATE).  
2. DAG + lista de supuestos.  
3. Por qué el adjustment set es defendible *aquí* y qué fallaría en datos reales.  
4. Qué dicen balance y overlap.  
5. Un límite de validez externa.


---

## 10. EVALUACIÓN CON GROUND TRUTH (solo al final)

> ⚠️ **SECCIÓN DE EVALUACIÓN — NO FORMA PARTE DEL FLUJO PRINCIPAL DEL ESTUDIANTE**  
> A partir de aquí se carga `truth_*.csv` (`y0`, `y1`, `tau`) **únicamente** para calcular errores absolutos del ATE y cerrar el ciclo pedagógico.  
> **No** uses estas columnas para re-entrenar ni para seleccionar el estimador en la entrega.  
> El autograder también usa truth de forma independiente.

---


In [ ]:
# === EVALUACIÓN CON TRUTH (marcada) ===
from lib.data_io import load_truth
from lib.metrics import ate_from_tau

# Truth en test (y opcionalmente train) — solo evaluación
truth_test = load_truth(SCENARIO_A1, "test")
truth_train = load_truth(SCENARIO_A1, "train")

assert "tau" in truth_test.columns
ate_true_meta = float(meta.get("ate_true"))
ate_true_from_tau_test = ate_from_tau(truth_test["tau"])
ate_true_from_tau_train = ate_from_tau(truth_train["tau"])

print("=== Ground truth (evaluación) ===")
print(f"ate_true (metadata)     : {ate_true_meta:,.4f}")
print(f"mean(tau) train         : {ate_true_from_tau_train:,.4f}")
print(f"mean(tau) test          : {ate_true_from_tau_test:,.4f}")

# Errores de nuestros estimadores vs metadata ATE
eval_estimates = {
    "dim_train": ate_dim_train,
    "ols_train": ate_ols_train,
    "pds_train": ate_pds_train,
    "dim_test": ate_dim_test,
    "ols_test": ate_ols_test,
    "pds_test": ate_pds_test,
}
eval_table = summarize_ate_comparison(eval_estimates, ate_true=ate_true_meta)
print("\n=== AE vs ate_true (metadata) ===")
print(eval_table.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

# Umbrales del curso (expected_ranges.yaml: max_abs_error_ols/pds = 250)
MAX_AE = 250.0
for name in ("ols_train", "pds_train", "ols_test", "pds_test"):
    ae = float(eval_table.loc[eval_table["estimator"] == name, "abs_error"].iloc[0])
    status = "OK" if ae < MAX_AE else "FAIL"
    print(f"  {name}: AE={ae:,.2f}  [{status} vs umbral {MAX_AE}]")

print("\nNota: un AE bajo en el SEM no sustituye la justificación escrita de identificación.")


## Checklist de entrega

- [ ] Bloque de transparencia leído; solo `observed_*` en estimación  
- [ ] Pregunta causal + estimando ATE declarados  
- [ ] DAG + supuestos (ignorabilidad, overlap, SUTVA)  
- [ ] Tabla de balance (SMD) y figura de propensity  
- [ ] Diff-in-means, OLS y PDS ejecutados  
- [ ] SE e IC 95% (diff-in-means, OLS, PDS) en la tabla comparativa (§8)  
- [ ] Tabla comparativa vs scorecard  
- [ ] Justificación escrita (§9) no vacía  
- [ ] Sección truth (§10) solo para evaluación final  

**Siguiente en la serie:** A2 (DML sobre `ihdp_medium`) — no requiere artefactos de A1.
